In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, probplot
import pandas as pd

In [ ]:
N_values = [10**3, 10**4, 10**5, 10**6]
C_lim = 7.0

results = []
for N in N_values:

    v, alpha_L, lamb = sample_parameters(N)
    C_peak = calculate_peak_concentration(v, alpha_L, lamb, M=solver.M, L2=solver.L2)

    failures = C_peak > C_lim
    N_fail = np.sum(failures)

    P_f = N_fail/N

    SE = np.sqrt(P_f * (1 - P_f) / N)
    RSE = SE / P_f if P_f > 0 else np.inf

    CI_lower = P_f - 1.96 * SE
    CI_upper = P_f + 1.96 * SE

    CI_lower = max(0, CI_lower)
    CI_upper = min(1, CI_upper)

    results.append({
        "N": N,
        "Failures": N_fail,
        "P_f": P_f,
        "SE": SE,
        "RSE": RSE,
        "CI_lower": CI_lower,
        "CI_upper": CI_upper
    })

results_df = pd.DataFrame(results)

print(results_df)

In [ ]:
P_f_est = results_df.iloc[-1]["P_f"]

if P_f_est > 0:
    N_required = (1 - P_f_est) / (P_f_est * 0.1**2)

    print(f"Estimated failure probability: {P_f_est:.6f}")
    print(f"Required samples for 10% RSE: {N_required:.0f}")
else:
    print("No failures were observed. Cannot estimate required sample size.")

In [ ]:
def midpoint_rule(n, M, L2=solver.L2, C_lim=7):
    # integration domain
    lower_bound = -5
    upper_bound = 5

    # midpoints
    dz = (upper_bound - lower_bound) / n
    z = lower_bound + (np.arange(n) + 0.5) * dz

    # standard normal pdf
    phi = norm.pdf(z)

    # standard normal variables
    v = np.exp(np.log(0.5) + 0.3 * z)
    alpha_L = np.exp(np.log(2) + 0.5 * z)
    lamb = np.exp(np.log(2e-3) + 0.3 * z)

    V, A, LAM = np.meshgrid(v, alpha_L, lamb, indexing='ij')

    C_peak = calculate_peak_concentration(V, A, LAM, M=M, L2=L2)

    indicator = C_peak > C_lim
    weights = (
        phi[:, None, None]
        * phi[None, :, None]
        * phi[None, None, :]
    )

    P_f = np.sum(indicator * weights) * dz**3

    return P_f

In [ ]:
n_values = [5, 10, 15, 20, 30, 40, 60, 80, 100, 150, 200]

midpoint_results = []

for n in n_values:
    P_f = midpoint_rule(n, M=solver.M, L2=500)
    midpoint_results.append(P_f)

    print(f"n = {n}, points = {n**3}, Pf = {P_f:.8f}")

plt.figure()
plt.plot(n_values, midpoint_results, "o-")

plt.xlabel("Number of intervals per dimension")
plt.ylabel("Failure probability")
plt.title("Midpoint rule convergence")
plt.show()

In [ ]:
def gauss_hermite(n, M, L2=solver.L2, C_lim=7):
    n = int(n)
    
    nodes, weights = np.polynomial.hermite_e.hermegauss(n)
    weights = weights/np.sqrt(2*np.pi)
    v = np.exp(np.log(0.5) + 0.3 * nodes)
    lamb = np.exp(np.log(2e-3) + 0.3 * nodes)

    V, LAM = np.meshgrid(v, lamb, indexing='ij')

    alpha_star = M**2 / (4 * np.pi * L2 * C_lim**2) * np.exp(-2 * LAM * L2 / V)

    conditional_Pf = norm.cdf(
        (np.log(alpha_star) - np.log(2)) / 0.5
    )

    P_f = np.sum(
        weights[:, None]
        * weights[None, :]
        * conditional_Pf 
    )

    return P_f

In [ ]:
gh_results = []

for n in n_values:

    P_f = gauss_hermite(
        n=n,
        M=solver.M,
        L2=500
    )

    gh_results.append(P_f)

    print(f"n = {n:g}, points = {n**2:g}, Pf = {P_f:.10f}")

plt.figure()
plt.plot(n_values, midpoint_results, "o-", color="blue", label="Midpoint rule")
plt.plot(n_values, gh_results, "o-", color="green", label="Gauss-Hermite")

plt.xlabel("Number of points per dimension (n)")
plt.ylabel("Failure probability")
plt.title("Comparison of numerical integration methods")
plt.legend()
plt.show()